# 05. MERGE INTO - 行ごとに「どうするか」を指示する

`04` では `replaceUsing` を扱いました。キー列が一致する行を **丸ごと置き換える** 仕組みです。

ただ実務では、もう少し細かい指示を出したくなります。

- 金額だけ更新したい。他の列は触りたくない
- 既にある行は更新、無い行は追加、という処理を1回でやりたい
- 上流から消えた行は、こちらでも消したい
- ある条件のときだけ更新したい

これらを行ごとに指示できるのが `MERGE INTO` です。

このノートブックで確かめること:

1. `MERGE` の基本形。一致したら更新、しなかったら追加
2. **列を選んで**更新できること (`replaceUsing` との違い)
3. `WHEN NOT MATCHED BY SOURCE` で、上流から消えた行を扱えること
4. 条件を付けて、更新するかどうかを選べること
5. **ソースにキーの重複があるとどうなるか**

**前提**: `00_setup` を実行済みであること。`04` を読んでいること。

## 準備

In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.silver.merge_orders"

## 1. テーブルを用意する

`04` と同じく Liquid Clustering で作ります。
今回は `status` 列を足しました。「一部の列だけ更新する」を試すためです。

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# DDL: クラスタリングキーは order_date にする
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        status STRING,
        order_date DATE
    )
    CLUSTER BY (order_date)
""")

# DML: 初期データを3件入れる
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000, 'placed', DATE '2026-09-11'),
        (2, 'monitor',   40000, 'placed', DATE '2026-09-11'),
        (3, 'keyboard',  12000, 'placed', DATE '2026-09-11')
""")

# 出発点の状態を確認する
display(spark.table(TABLE).orderBy("order_id"))

## 2. `MERGE` の基本形

`MERGE` は「2つの表を突き合わせて、行ごとに処理を決める」命令です。構造はこうなっています。

```sql
MERGE INTO 更新したい表 AS t        -- 変更される側 (ターゲット)
USING 新しいデータ AS s             -- 変更する材料 (ソース)
ON t.order_id = s.order_id          -- どう突き合わせるか
WHEN MATCHED THEN ...               -- 両方にあった行をどうするか
WHEN NOT MATCHED THEN ...           -- ソースにだけあった行をどうするか
```

まず「あれば更新、なければ追加」をやります。
ソースには **既存のorder_id=1 と、存在しないorder_id=4** を混ぜてあります。

In [ ]:
# 更新1件 (order_id=1) と 新規1件 (order_id=4) を含むソース
spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING (
        SELECT * FROM VALUES
            (1, 'laptop',  155000, 'shipped', DATE '2026-09-11'),
            (4, 'mouse',     5000, 'placed',  DATE '2026-09-11')
        AS s(order_id, product, amount, status, order_date)
    ) AS s
    ON t.order_id = s.order_id

    -- 両方にある行: 全列をソースの値で上書きする
    WHEN MATCHED THEN UPDATE SET *

    -- ソースにしかない行: そのまま追加する
    WHEN NOT MATCHED THEN INSERT *
""")

# order_id=1 が更新され、order_id=4 が増えていることを確認する
display(spark.table(TABLE).orderBy("order_id"))

`UPDATE SET *` と `INSERT *` は「全列をまとめて」という書き方です。
ここまでは `replaceUsing` でもほぼ同じことができます。

## 3. 列を選んで更新する

ここからが `MERGE` の本領です。

「金額だけ直したい。ステータスは今の値を保ちたい」という状況を考えます。
`replaceUsing` は行を丸ごと置き換えるので、この指示は出せません。
`MERGE` なら `UPDATE SET` で更新する列を選べます。

order_id=1 は今 `shipped` になっています。金額だけ変えて、これが残るかを見ます。

In [ ]:
spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING (
        SELECT * FROM VALUES
            (1, 160000)
        AS s(order_id, amount)
    ) AS s
    ON t.order_id = s.order_id

    -- amount だけ更新する。status や product には触らない
    WHEN MATCHED THEN UPDATE SET t.amount = s.amount
""")

# order_id=1 の amount だけが変わり、status が shipped のままであることを確認する
display(spark.table(TABLE).orderBy("order_id"))

ソースに `amount` しか持たせていない点にも注目してください。
突き合わせに使う `order_id` と、更新したい列さえあれば動きます。
全列を揃える必要がありません。

## 4. 上流から消えた行をどうするか

もう1つ、`MERGE` にしかできないことがあります。

**ソースに無かったターゲットの行** を処理する `WHEN NOT MATCHED BY SOURCE` です。

状況を考えます。上流が「9月11日の注文はこの2件が全てです」と言ってきたとします。
ターゲットには4件あるので、残り2件は取り消されたことになります。

| 句 | 対象 |
|---|---|
| `WHEN MATCHED` | 両方にある行 |
| `WHEN NOT MATCHED` | **ソースにだけ**ある行 |
| `WHEN NOT MATCHED BY SOURCE` | **ターゲットにだけ**ある行 |

実行前に、4件のうちどれが残るか予想してください。

In [ ]:
spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING (
        SELECT * FROM VALUES
            (1, 'laptop',   160000, 'shipped', DATE '2026-09-11'),
            (2, 'monitor',   40000, 'placed',  DATE '2026-09-11')
        AS s(order_id, product, amount, status, order_date)
    ) AS s
    ON t.order_id = s.order_id

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *

    -- ソースに無かった行 = 上流で取り消された行とみなして消す
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

# ソースにあった2件だけが残っていることを確認する
display(spark.table(TABLE).orderBy("order_id"))

`04` で見た `replaceWhere` も「範囲ごと入れ替える」ので似た結果になりますが、考え方が違います。

- `replaceWhere` … **範囲を条件で宣言**して、その中身を丸ごと差し替える
- `WHEN NOT MATCHED BY SOURCE` … **行ごとに突き合わせて**、余った行を処理する

後者は「消す」以外も選べます。`DELETE` の代わりに
`UPDATE SET t.status = 'cancelled'` と書けば、消さずに印を付けられます。
履歴を残したい場合はこちらになります。

## 5. 条件を付ける

`WHEN MATCHED` には条件を足せます。「一致していて、かつ〜のときだけ」という指定です。

よくあるのは、**古い情報で新しい情報を上書きしないようにする** 使い方です。
順序が前後して届いたデータで、せっかくの更新を巻き戻してしまう事故を防げます。

ここでは「すでに `shipped` になっている注文は、`placed` に戻さない」を表現します。

In [ ]:
spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING (
        SELECT * FROM VALUES
            (1, 'placed'),
            (2, 'shipped')
        AS s(order_id, status)
    ) AS s
    ON t.order_id = s.order_id

    -- ターゲットが shipped のときは更新しない。それ以外なら更新する
    WHEN MATCHED AND t.status <> 'shipped' THEN UPDATE SET t.status = s.status
""")

# order_id=1 は shipped のまま、order_id=2 だけが shipped に変わることを確認する
display(spark.table(TABLE).orderBy("order_id"))

## 6. ソースにキーの重複があると

`MERGE` を使う上で最も引っかかりやすい点です。

ソース側に同じキーの行が2つあると、ターゲットの1行に対して2つの更新候補ができます。
Deltaはどちらを採用すべきか決められません。どうなるか試します。

In [ ]:
try:
    spark.sql(f"""
        MERGE INTO {TABLE} AS t
        USING (
            SELECT * FROM VALUES
                (1, 170000),
                (1, 180000)
            AS s(order_id, amount)
        ) AS s
        ON t.order_id = s.order_id
        WHEN MATCHED THEN UPDATE SET t.amount = s.amount
    """)
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

エラーになったはずです。

黙ってどちらかが採用されるより、はるかに親切な動作です。
「なぜか金額が実行するたびに変わる」という追いにくい不具合になるところを、実行時点で止めてくれます。

実務ではソースに重複が入り込むことがよくあります。
上流が同じレコードを2回送ってきた、結合で行が増えた、などです。
`MERGE` の前に **キーで重複を排除しておく** のが定石になります。
「どれを残すか」(最新のタイムスタンプのものを残す、など) は自分で決める必要があります。

## 7. `replaceUsing` との使い分け

| | `replaceUsing` (04) | `MERGE INTO` (05) |
|---|---|---|
| 更新の単位 | 行を丸ごと置き換える | **列を選んで**更新できる |
| ソースに無い列 | 揃える必要がある | キーと更新する列だけでよい |
| ターゲットにだけある行 | 触らない | `NOT MATCHED BY SOURCE` で削除も更新もできる |
| 条件付きの更新 | できない | `WHEN MATCHED AND ...` で指定できる |
| ソースのキー重複 | エラーにならない | **エラーになる** |
| 書き方 | DataFrame API のオプション | SQL (または DeltaTable API) |

できることは `MERGE` のほうが多いですが、その分「何が起きるか」を自分で決める必要があります。

「届いた行で丸ごと置き換えればよい」だけなら `replaceUsing` のほうが短く書け、
意図も明確になります。列単位の制御や削除の伝播が要るときに `MERGE` を選ぶ、
という順序で考えるとよさそうです。

## 考えてみる

- `3.` でソースに `order_id` と `amount` しか持たせませんでした。これが `replaceUsing` ではできないのはなぜでしょうか
- `4.` で `DELETE` ではなく `UPDATE SET t.status = 'cancelled'` にすると、何が嬉しいでしょうか
- `6.` の重複を排除するとき、「どれを残すか」はどう決めればよいでしょうか

### 答え

**Q1. なぜ `replaceUsing` では列を選べないのか**

`replaceUsing` は **行の置き換え** だからです。
キーが一致した行を消して、ソースの行をそのまま入れる、という動作をします。
ソースに `status` が無ければ、置き換わった後の行にも `status` はありません。

`MERGE` の `UPDATE SET` は **既存の行を書き換える** 操作です。
指定しなかった列は元の値のまま残ります。同じ「更新」でも、成り立ちが違います。

**Q2. 消さずに印を付ける利点**

3つあります。

- **後から追える**。いつ取り消されたのか、何が取り消されたのかが残る
- **下流に伝わる**。行が消えるだけだと、下流は「無くなったこと」に気づきにくい。
  `cancelled` という行が来れば、`03` で見たCDFの変更として拾えます
- **戻せる**。取り消しが誤りだった場合に、status を戻すだけで復旧できる

一方で、行が増え続けるので、いつか整理する仕組みは要ります。

**Q3. 重複のうちどれを残すか**

データの性質によるので、決まった正解はありません。よく使うのは次の考え方です。

- **更新時刻の列があるなら、最も新しいもの**。これが最も素直です
- 時刻が無いなら、**上流のシーケンス番号やファイルの到着順**
- どちらも無い場合は、そもそも「どれが正しいか」を決められません。
  上流に順序が分かる情報を足してもらうのが本筋になります

「とりあえず1件に絞る」とだけ決めて適当に選ぶと、実行するたびに結果が変わる
不安定な処理になります。`06_idempotent_writes` で扱う「何度実行しても同じ結果になる」
という性質が、ここで壊れます。

## 後片付け

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")